In [1]:
import pylhe
import numpy as np
import matplotlib.pyplot as plt

In [13]:
data = pylhe.read_lhe('../Events/run_01/unweighted_events.lhe.gz')

In [3]:
p_w=[]

for evt in data:
    for particle in evt.particles:
        status= getattr(particle,"status")
        idd   = getattr(particle,"id")
        if status ==2:                             #w+ is an intermediate particle
            if idd == 24 : #top
                E  =getattr(particle,"e")
                px =getattr(particle,"px")
                py =getattr(particle,"py")
                pz =getattr(particle,"pz")
                p =[E,px,py,pz]
                p_w.append(p)
                

/tmp/ipykernel_14226/3520403093.py:3: DeprecationWarning: read_lhe is deprecated and will be removed in a future version. Use `LHEFile.fromfile(...).events` instead
  for evt in data:


In [4]:
pw_eg = p_w[0]
print(pw_eg)

phi = np.arctan2(pw_eg[2], pw_eg[1])
print(phi*(180/np.pi))

theta = np.arccos(pw_eg[3]/np.sqrt(pw_eg[1]**2 + pw_eg[2]**2 + pw_eg[3]**2))
print(theta*(180/np.pi))

[125.33204074, -44.086395545, 49.464229364, 69.19861405]
131.70992643656632
43.75701141874434


In [5]:
def rotate_z(p, phi):
    c = np.cos(phi)
    s = np.sin(phi)
    return [p[0], c*p[1] - s*p[2], s*p[1] + c*p[2], p[3]]

In [6]:
# def calc_phi(px, py):
#     if px>0 and py>0:              #1st quadrant
#         return np.arctan(py/px)
#     if px>0 and py<0:
#         return np.arctan(py/px)
#     if px<0 and py>0:
#         return np.pi + np.arctan(py/px)
#     if px<0 and py<0:
#         return np.pi + np.arctan(py/px)
    

In [7]:
pw_eg = rotate_z(pw_eg, -phi)
print(pw_eg)

[125.33204074, np.float64(66.25949183871391), np.float64(-1.4210854715202004e-14), 69.19861405]


In [8]:
def rotate_y(p, theta):
    c = np.cos(theta)
    s = np.sin(theta)
    return [p[0], c*p[1]+s*p[3], p[2], -s*p[1]+c*p[3]]

In [9]:
pw_eg = rotate_y(pw_eg, -theta)
print(pw_eg)

[125.33204074, np.float64(-1.4210854715202004e-14), np.float64(-1.4210854715202004e-14), np.float64(95.8058894075174)]


Now, we boost

In [10]:
def beta(p):
    p_mag = np.sqrt(p[1]**2 + p[2]**2 + p[3]**2)
    return p_mag/p[0]

def gamma(p):
    p_mag = np.sqrt(p[1]**2 + p[2]**2 + p[3]**2)
    beta = p_mag/p[0]
    return 1.0 / np.sqrt(1.0 - beta**2)

In [11]:
#Boost function
def boost_z(p):
    p_n = [0,0,0,0]
    p_n[0]  = gamma(p) * (p[0] - beta(p) * p[3])
    p_n[3] = gamma(p) * (p[3] - beta(p) * p[0])
    p_n[1] = p[1]
    p_n[2] = p[2]
    return p_n

In [12]:
pw_eg = boost_z(pw_eg)
print(pw_eg)

[np.float64(80.80440576408915), np.float64(-1.4210854715202004e-14), np.float64(-1.4210854715202004e-14), np.float64(0.0)]


In [14]:
p_e = []
p_v = []
p_w = []

for evt in data:
    for particle in evt.particles:
        status= getattr(particle,"status")
        idd   = getattr(particle,"id")
        if status == 2:                             #w is an intermediate particle
            if idd == 24 : #w+
                E  =getattr(particle,"e")
                px =getattr(particle,"px")
                py =getattr(particle,"py")
                pz =getattr(particle,"pz")
                p =[E,px,py,pz]
                p_w.append(p)
        
        if status == 1:                             #final state particle
            if idd == -11 : #e+
                E  =getattr(particle,"e")
                px =getattr(particle,"px")
                py =getattr(particle,"py")
                pz =getattr(particle,"pz")
                p =[E,px,py,pz]
                p_e.append(p)

        if status == 1:                             #final state particle
            if idd == 12 : #ve
                E  =getattr(particle,"e")
                px =getattr(particle,"px")
                py =getattr(particle,"py")
                pz =getattr(particle,"pz")
                p =[E,px,py,pz]
                p_v.append(p)

print(len(p_w))
print(len(p_e))
print(len(p_v))

/tmp/ipykernel_14226/2410751914.py:5: DeprecationWarning: read_lhe is deprecated and will be removed in a future version. Use `LHEFile.fromfile(...).events` instead
  for evt in data:


1000000
1000000
1000000


In [15]:
def boost_z_daughter(pm, pd):
    pd_n = [0,0,0,0]
    pd_n[0] = gamma(pm) * (pd[0] - beta(pm) * pd[3])
    pd_n[3] = gamma(pm) * (pd[3] - beta(pm) * pd[0])
    pd_n[1] = pd[1]
    pd_n[2] = pd[2]
    return pd_n

In [16]:
def transformRestofMother(mother, daughter):
    pd_r = []
    for i in range(0, len(mother)):
        phi = np.arctan2(mother[i][2], mother[i][1])
        theta = np.arccos(mother[i][3]/np.sqrt(mother[i][1]**2 + mother[i][2]**2 + mother[i][3]**2))
        inter_pm = rotate_z(mother[i], -phi)
        inter_pd = rotate_z(daughter[i], -phi)
        inter_pm = rotate_y(inter_pm, -theta)
        inter_pd = rotate_y(inter_pd, -theta)
        #Boosting
        pd_r.append(boost_z_daughter(inter_pm, inter_pd))
    
    return pd_r    


In [17]:
#Then the rest frame particle vectors
pe_r = transformRestofMother(p_w, p_e)
pv_r = transformRestofMother(p_w, p_v)
pt_r = transformRestofMother(p_w, p_w)

#Verification
print(len(pe_r))
#print(pt_r[0])
#print(pt_r[10])

1000000


In [18]:
print(pt_r[0])
print(pt_r[10])

[np.float64(80.80440576408915), np.float64(-1.4210854715202004e-14), np.float64(-1.4210854715202004e-14), np.float64(0.0)]
[np.float64(81.78022288291159), np.float64(0.0), np.float64(3.552713678800501e-15), np.float64(0.0)]


In [19]:
pw_r = pt_r
print(len(pw_r))

1000000


In [20]:
import uproot

In [21]:
w  = np.array(pw_r)
e  = np.array(pe_r)
v = np.array(pv_r)

In [22]:
with uproot.recreate("w+_restframe.root") as f:
    
    f["Events"] = {
        "w_E":  w[:,0],
        "w_px": w[:,1],
        "w_py": w[:,2],
        "w_pz": w[:,3],

        "e_E":  e[:,0],
        "e_px": e[:,1],
        "e_py": e[:,2],
        "e_pz": e[:,3],

        "ve_E":  v[:,0],
        "ve_px": v[:,1],
        "ve_py": v[:,2],
        "ve_pz": v[:,3],
    }

## Plotter 

Now, we wish to plot $\phi_f$ and $\cos{\theta_f}$ in the rest frame

In [ ]:
# def plotter(p, pname):
#     phi_f = []
#     cos_theta_f = []

#     for i in range(0, len(p)):
#         phi = np.arctan2(p[i][2], p[i][1])
#         theta = np.arccos(
#             p[i][3] / np.sqrt(p[i][1]**2 + p[i][2]**2 + p[i][3]**2)
#         )
#         cos_theta = np.cos(theta)
#         phi_f.append(phi)
#         cos_theta_f.append(cos_theta)

#     plt.figure()
#     plt.hist(phi_f, bins = 50)
#     plt.title(f"Phi distribution of {pname} in rest frame of mother particle")
#     plt.xlabel("Phi")
#     plt.ylabel("Counts")
#     plt.savefig(f"plots/rest_frame_of_top/top~_channel/phi_{pname}.png")
#     plt.close()

#     plt.figure()
#     plt.hist(cos_theta_f, bins = 50)
#     plt.title(f"Cos theta distribution of {pname} in rest frame of mother particle")
#     plt.xlabel("Cos theta")
#     plt.ylabel("Counts")
#     plt.savefig(f"plots/rest_frame_of_top/top~_channel/costheta_{pname}.png")
#     plt.close()    

In [ ]:

# plotter(pe_r, "mu-")
# plotter(pv_r, "vm~")
# plotter(pb_r, "b~")

Now, we plot $\frac{m_{ij}}{2m_t}$

In [ ]:
# def plotter2(pt, pi, pj, pname1, pname2):
#     data = []
    
#     for i in range(0, len(pt)):
#         mt = np.sqrt(pt[i][0]**2 - pt[i][1]**2 - pt[i][2]**2 - pt[i][3]**2)
#         mij = np.sqrt((pi[i][0] + pj[i][0])**2 - (pi[i][1] + pj[i][1])**2 - (pi[i][2] + pj[i][2])**2 - (pi[i][3] + pj[i][3])**2)
#         data.append(mij/(2*mt))

#     plt.figure()
#     plt.hist(data, bins = 50)
#     plt.title(f"mij/2mt for {pname1}, {pname2}")
#     plt.xlabel("mij/2mt")
#     plt.ylabel("Counts")
#     plt.savefig(f"plots/mij_by_2mt/top~_channel/{pname1}_{pname2}.png")
#     plt.close() 

In [ ]:
# plotter2(pt_r, pe_r, pv_r, "mu-", "vm~")
# plotter2(pt_r, pe_r, pb_r, "mu-", "b~")
# plotter2(pt_r, pb_r, pv_r, "b~", "vm~")